<a href="https://colab.research.google.com/github/intanoktavianipresia/Studi_Kasus_Deep_Learning/blob/main/Sesi_3_GIS_Pemetaan_Rawan_Banjir.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Studi Kasus GIS: Pemetaan Awal Daerah Potensi Banjir

## Tujuan
1. Mengambil data geografis nyata dari OpenStreetMap.
2. Menampilkan jaringan sungai dan saluran air.
3. Mengidentifikasi fitur geografis terkait banjir.
4. Memvisualisasikan data menggunakan peta interaktif.

## Sumber Data
OpenStreetMap melalui Overpass API.

Catatan: Hasil pemetaan merupakan analisis awal,
bukan peta resmi risiko banjir.

In [3]:
!pip -q install requests pandas folium

In [4]:
import requests
import pandas as pd
import folium
import time
from IPython.display import display

# Wilayah studi: Bengkulu Utara, Bengkulu, Indonesia
# Batas geografis perkiraan untuk pengambilan data
SOUTH, WEST, NORTH, EAST = -3.7, 101.4, -2.7, 102.4

# Overpass API menyediakan data geografis OpenStreetMap
endpoints = [
    "https://overpass-api.de/api/interpreter",
    "https://overpass.kumi.systems/api/interpreter"
]

query = f"""
[out:json][timeout:60];
(
  way["waterway"~"river|stream|canal|drain"]
    ({SOUTH},{WEST},{NORTH},{EAST});
  node["hazard"="flood"]
    ({SOUTH},{WEST},{NORTH},{EAST});
  way["hazard"="flood"]
    ({SOUTH},{WEST},{NORTH},{EAST});
  node["natural"="water"]
    ({SOUTH},{WEST},{NORTH},{EAST});
);
out body geom;
"""

data = None
error_terakhir = None

for endpoint in endpoints:
    try:
        print("Mengakses sumber data:", endpoint)

        response = requests.post(
            endpoint,
            data={"data": query},
            timeout=90,
            headers={"User-Agent": "Colab-GIS-Educational-Project"}
        )

        response.raise_for_status()
        hasil = response.json()

        if "elements" not in hasil:
            raise ValueError("Respons tidak memiliki elemen geografis.")

        data = hasil["elements"]
        print("Data berhasil diambil.")
        break

    except Exception as error:
        error_terakhir = error
        print("Endpoint gagal:", str(error))
        time.sleep(2)

if data is None:
    raise RuntimeError(
        "Sumber data belum dapat diakses. "
        "Coba jalankan kembali beberapa saat lagi."
    )

print("Jumlah objek geografis:", len(data))

Mengakses sumber data: https://overpass-api.de/api/interpreter
Data berhasil diambil.
Jumlah objek geografis: 727


In [5]:
records = []

for item in data:
    tags = item.get("tags", {})
    geometry = item.get("geometry", [])

    if item["type"] == "node":
        lat = item.get("lat")
        lon = item.get("lon")
        koordinat = [(lat, lon)] if lat is not None and lon is not None else []

    else:
        koordinat = [
            (point["lat"], point["lon"])
            for point in geometry
            if "lat" in point and "lon" in point
        ]

    if not koordinat:
        continue

    records.append({
        "id": item["id"],
        "tipe_objek": item["type"],
        "nama": tags.get("name", "Tidak bernama"),
        "jenis_air": tags.get("waterway", ""),
        "jenis_bahaya": tags.get("hazard", ""),
        "jenis_perairan": tags.get("natural", ""),
        "jumlah_titik": len(koordinat),
        "koordinat": koordinat,
        "latitude": koordinat[0][0],
        "longitude": koordinat[0][1]
    })

df = pd.DataFrame(records)

if df.empty:
    raise ValueError(
        "Tidak ditemukan objek yang sesuai dalam batas wilayah ini. "
        "Periksa kembali wilayah atau coba lagi nanti."
    )

print("Jumlah fitur yang berhasil diproses:", len(df))
display(df.drop(columns=["koordinat"]).head(10))

Jumlah fitur yang berhasil diproses: 727


,id,tipe_objek,nama,jenis_air,jenis_bahaya,jenis_perairan,jumlah_titik,latitude,longitude
0,136094126,way,Tidak bernama,stream,,,10,-3.461599,102.127750
1,136094127,way,Air Nokan,river,,,45,-3.418083,102.189391
2,136094131,way,Sungai Lemau,river,,,176,-3.644934,102.261422
3,136094133,way,Air Lais,river,,,90,-3.425758,102.165209
4,136094134,way,Tidak bernama,canal,,,113,-3.350878,102.138936
5,136094135,way,Air Nokan,river,,,14,-3.428221,102.169331
6,136094136,way,Air Lais,river,,,96,-3.481569,102.091301
7,136094139,way,Tidak bernama,river,,,27,-3.464346,102.135110
8,136094143,way,Tidak bernama,river,,,27,-3.421656,102.197391
9,136094147,way,Tidak bernama,stream,,,68,-3.464786,102.132522


In [6]:
# Menghitung fitur jaringan air
jumlah_jaringan_air = (df["jenis_air"] != "").sum()

# Menghitung objek yang secara eksplisit memiliki tag hazard=flood
jumlah_tag_banjir = (df["jenis_bahaya"] == "flood").sum()

# Menghitung fitur perairan alami
jumlah_perairan = (df["jenis_perairan"] == "water").sum()

ringkasan = pd.DataFrame({
    "Kategori": [
        "Jaringan sungai dan saluran air",
        "Objek bertanda bahaya banjir",
        "Perairan alami"
    ],
    "Jumlah": [
        int(jumlah_jaringan_air),
        int(jumlah_tag_banjir),
        int(jumlah_perairan)
    ]
})

display(ringkasan)

print(
    "\nInterpretasi:\n"
    "- Jaringan air menunjukkan fitur hidrologi yang dipetakan.\n"
    "- Tag banjir hanya menunjukkan objek yang ditandai demikian "
    "oleh kontributor OpenStreetMap.\n"
    "- Jumlah objek bukan ukuran tingkat risiko banjir."
)

,Kategori,Jumlah
0,Jaringan sungai dan saluran air,727
1,Objek bertanda bahaya banjir,0
2,Perairan alami,0



Interpretasi:
- Jaringan air menunjukkan fitur hidrologi yang dipetakan.
- Tag banjir hanya menunjukkan objek yang ditandai demikian oleh kontributor OpenStreetMap.
- Jumlah objek bukan ukuran tingkat risiko banjir.


In [7]:
peta = folium.Map(
    location=[(SOUTH + NORTH) / 2, (WEST + EAST) / 2],
    zoom_start=9,
    tiles="OpenStreetMap"
)

layer_air = folium.FeatureGroup(
    name="Sungai dan saluran air",
    show=True
)

layer_banjir = folium.FeatureGroup(
    name="Objek bertanda banjir",
    show=True
)

layer_perairan = folium.FeatureGroup(
    name="Perairan alami",
    show=True
)

for _, row in df.iterrows():
    coords = row["koordinat"]
    nama = row["nama"]

    if row["jenis_air"]:
        # Data garis sungai/saluran air
        if len(coords) >= 2:
            folium.PolyLine(
                locations=coords,
                tooltip=f"{nama} — {row['jenis_air']}",
                weight=3
            ).add_to(layer_air)

        else:
            folium.CircleMarker(
                location=coords[0],
                radius=4,
                tooltip=f"{nama} — {row['jenis_air']}"
            ).add_to(layer_air)

    if row["jenis_bahaya"] == "flood":
        folium.Marker(
            location=[row["latitude"], row["longitude"]],
            tooltip=f"Tag banjir: {nama}",
            popup="Objek OpenStreetMap dengan tag hazard=flood"
        ).add_to(layer_banjir)

    if row["jenis_perairan"] == "water":
        folium.CircleMarker(
            location=[row["latitude"], row["longitude"]],
            radius=4,
            tooltip=f"Perairan: {nama}"
        ).add_to(layer_perairan)

layer_air.add_to(peta)
layer_banjir.add_to(peta)
layer_perairan.add_to(peta)

folium.LayerControl().add_to(peta)

display(peta)

In [10]:
# Simpan tabel hasil ke CSV
df.drop(columns=["koordinat"]).to_csv(
    "hasil_pemetaan_gis_bengkulu_utara.csv",
    index=False
)

# Simpan peta interaktif ke HTML
peta.save("peta_gis_bengkulu_utara.html")

print("File berhasil disiapkan:")
print("- hasil_pemetaan_gis_bengkulu_utara.csv")
print("- peta_gis_bengkulu_utara.html")

File berhasil disiapkan:
- hasil_pemetaan_gis_bengkulu_utara.csv
- peta_gis_bengkulu_utara.html


In [9]:
from google.colab import files

files.download("hasil_pemetaan_gis_bengkulu_utara.csv")
files.download("peta_gis_bengkulu_utara.html")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

## Kesimpulan

Studi kasus ini menggunakan GIS untuk mengambil, mengolah,
dan memvisualisasikan data geografis nyata dari OpenStreetMap.

Peta menampilkan jaringan sungai, saluran air, dan objek yang
memiliki penandaan terkait banjir jika tersedia dalam sumber data.

Peta ini belum dapat menentukan daerah yang benar-benar rawan banjir.
Analisis risiko yang lebih akurat memerlukan data kejadian banjir
historis, elevasi, kemiringan lahan, curah hujan, serta validasi
dari sumber resmi.